# Training Notebook

This notebook contains the models we have trained, in increasing order of complexity with enhancements to the base Bi-LSTM model. 

To run this notebook, place the 4 data files, the 2 glove embedding files, and the 4 model code files in the same directory/workspace as this notebook:
* test.json
* train.json
* test-00000-of-00001.parquet
* train-00000-of-00001.parquet
* glove.6B.300d.txt
* glove.6B.100d.txt
* Base_BLSTM.py
* GloveEmbeddings_BLSTM.py
* Att_BLSTM.py
* MultiHeadAtt_BLSTM.py

As long as the imported packages from the first cell are imported correctly, and the above files are present, the entire notebook should run. It will train every model, and save each model as a .plk file, which can then be used with the InferenceBiLSTM.ipynb notebook. 

These .plk files have been supplied with our submission so it is not necessary to run this notebook to generate these models.


## Import Required Packages

In [1]:
#Importing required libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.nn import init
import torch.nn.functional as F
import pandas as pd
import random
import warnings
warnings.filterwarnings('ignore')

## Base Bi-LSTM Model

In [2]:
from Base_BLSTM import RelationClassification_Model, Vocabulary_and_Dataframe, get_relation_from_id, SemEval2010_task8_Dataset, batch_collate_fn, train_model, test_model

In [3]:
train_df = pd.read_parquet('train-00000-of-00001.parquet', engine='pyarrow')
test_df = pd.read_parquet('test-00000-of-00001.parquet', engine='pyarrow')

vocab_df = Vocabulary_and_Dataframe(freq_threshold=5)

vocab_df.build_vocabulary_and_dataframe(train_df)
vocab_df.build_vocabulary_and_dataframe(test_df)
train_df = train_df.assign(relationlabels=lambda x: x.relation.apply(get_relation_from_id))
test_df = test_df.assign(relationlabels=lambda x: x.relation.apply(get_relation_from_id))

In [10]:
indices = list(range(8000))
train_indices = indices
random.shuffle(indices)
test_indices = test_df.index.values
print(len(train_indices))
print(len(test_indices))

partition = {
    'train' : train_indices, 
    'test' : test_indices 
}

training_set = SemEval2010_task8_Dataset(train_df,partition['train'],vocab_df)
testing_set =  SemEval2010_task8_Dataset(train_df,partition['test'],vocab_df)

vocab_size = len(vocab_df)
pad_idx = vocab_df.str_2_idx[""]
class_num = 19

8000
2717


In [5]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
#device = torch.device("cpu")
torch.backends.cudnn.benchmark = True

In [6]:
torch.manual_seed(2048)

emb_size = 100
hidden = 50
layers = 1
lr = 0.005
wd = 0.0001
num_epochs = 50

params = {'batch_size': 256,
          'shuffle': True,
          'num_workers':0,
          'collate_fn' : batch_collate_fn(pad_idx=pad_idx)
          }
training_generator = torch.utils.data.DataLoader(training_set, **params)

model = RelationClassification_Model(emb_size, hidden, vocab_size, layers, class_num,pad_idx).to(device)

# Define Loss function and Optimizer
criterion = nn.NLLLoss(ignore_index = vocab_df.str_2_idx[""])
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

In [7]:
print("Training model...")
model_stats = train_model(model, optimizer, criterion, num_epochs, training_generator, device)
testing_stats = test_model(test_df, partition, vocab_df, device, model, len(partition['test']))
output_text = [emb_size, hidden, layers, lr, wd, num_epochs] + testing_stats
print(output_text)


Training model...


100%|██████████| 50/50 [00:17<00:00,  2.88it/s]


[100, 50, 1, 0.005, 0.0001, 50, 0.5666036266529164, 0.5772601923121222, 0.6404122193595878, 0.5681986176062351, array([ 8,  2, 14, ...,  8,  2,  8], dtype=int64)]


In [8]:
torch.save(model.state_dict(), f'model_base.pkl')

## Glove Embeddings Enhanced Bi-LSTM Model

In [6]:
from GloveEmbeddings_BLSTM import load_glove_embeddings

In [17]:
embedding_size = 300
hidden_size = 100
num_layers = 2
learning_rate = 0.005
weight_decay= 0.0001
num_epochs = 100

print("Loading Glove embeddings...")
glove_embeddings = load_glove_embeddings(embedding_size, vocab_df)

params = {'batch_size': 256,
          'shuffle': True,
          'num_workers':0,
          'collate_fn' : batch_collate_fn(pad_idx=pad_idx)
          }
training_generator = torch.utils.data.DataLoader(training_set, **params)

glove_model = RelationClassification_Model(embedding_size, hidden_size, vocab_size, num_layers, class_num, pad_idx)
glove_model.embedding = nn.Embedding.from_pretrained(torch.tensor(glove_embeddings, dtype=torch.float), freeze=False)
glove_model.to(device)

# Define Loss function and Optimizer
criterion = nn.NLLLoss(ignore_index = vocab_df.str_2_idx[""])
optimizer = optim.Adam(glove_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print("Training model...")
model_stats = train_model(glove_model, optimizer, criterion, num_epochs, training_generator, device)
testing_stats = test_model(test_df, partition, vocab_df, device, glove_model, len(partition['test']))
output_text = [embedding_size, hidden_size, num_layers, learning_rate, weight_decay, num_epochs] + testing_stats
print(output_text)

Loading Glove embeddings...
Training model...


100%|██████████| 100/100 [00:42<00:00,  2.33it/s]

[300, 100, 2, 0.005, 0.0001, 100, 0.6051910301610114, 0.5657315377631797, 0.6260581523739418, 0.5779028786373451, array([18, 17, 17, ..., 18,  2,  0], dtype=int64)]


In [18]:
torch.save(glove_model.state_dict(), f'model_glove.pkl')

## Attention Enhanced Bi-LSTM Model

In [12]:
from Att_BLSTM import WordEmbeddingLoader, RelationLoader, SemEvalDataLoader, Att_BLSTM, train

In [13]:
word2id, word_vec = WordEmbeddingLoader().load_embedding()
rel2id, id2rel, class_num = RelationLoader().get_relation()
loader = SemEvalDataLoader(rel2id, word2id)

train_loader = loader.get_train()
test_loader = loader.get_test()

loader = [train_loader, test_loader]

In [14]:
model = Att_BLSTM(word_vec=word_vec, class_num=class_num)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

In [15]:
train(model, criterion, loader, device)

Att_BLSTM(
  (word_embedding): Embedding(400006, 100)
  (lstm): LSTM(100, 100, batch_first=True, bidirectional=True)
  (tanh): Tanh()
  (emb_dropout): Dropout(p=0.3, inplace=False)
  (lstm_dropout): Dropout(p=0.3, inplace=False)
  (linear_dropout): Dropout(p=0.5, inplace=False)
  (dense): Linear(in_features=100, out_features=19, bias=True)
)
Training model parameters:
att_weight :  torch.Size([1, 100, 1])
word_embedding.weight :  torch.Size([400006, 100])
lstm.weight_ih_l0 :  torch.Size([400, 100])
lstm.weight_hh_l0 :  torch.Size([400, 100])
lstm.bias_ih_l0 :  torch.Size([400])
lstm.bias_hh_l0 :  torch.Size([400])
lstm.weight_ih_l0_reverse :  torch.Size([400, 100])
lstm.weight_hh_l0_reverse :  torch.Size([400, 100])
lstm.bias_ih_l0_reverse :  torch.Size([400])
lstm.bias_hh_l0_reverse :  torch.Size([400])
dense.weight :  torch.Size([19, 100])
dense.bias :  torch.Size([19])
--------------------------------------
Starting to train the model ...
[001/030] train_loss: 1.153 | test_loss: 1.1

## Multi-head Attention Enhanced Bi-LSTM Model

In [16]:
from MultiHeadAtt_BLSTM import WordEmbeddingLoader, RelationLoader, SemEvalDataLoader, MultiHeadAtt_BLSTM, train

In [17]:
word2id, word_vec = WordEmbeddingLoader().load_embedding()
rel2id, id2rel, class_num = RelationLoader().get_relation()
loader = SemEvalDataLoader(rel2id, word2id)

train_loader = loader.get_train()
test_loader = loader.get_test()

loader = [train_loader, test_loader]

In [18]:
model = MultiHeadAtt_BLSTM(word_vec=word_vec, class_num=class_num)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

In [19]:
train(model, criterion, loader, device)

MultiHeadAtt_BLSTM(
  (word_embedding): Embedding(400006, 100)
  (lstm): LSTM(100, 100, batch_first=True, bidirectional=True)
  (multihead_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
  )
  (tanh): Tanh()
  (emb_dropout): Dropout(p=0.3, inplace=False)
  (lstm_dropout): Dropout(p=0.3, inplace=False)
  (linear_dropout): Dropout(p=0.5, inplace=False)
  (dense): Linear(in_features=100, out_features=19, bias=True)
)
Training model parameters:
att_weight :  torch.Size([1, 100, 1])
word_embedding.weight :  torch.Size([400006, 100])
lstm.weight_ih_l0 :  torch.Size([400, 100])
lstm.weight_hh_l0 :  torch.Size([400, 100])
lstm.bias_ih_l0 :  torch.Size([400])
lstm.bias_hh_l0 :  torch.Size([400])
lstm.weight_ih_l0_reverse :  torch.Size([400, 100])
lstm.weight_hh_l0_reverse :  torch.Size([400, 100])
lstm.bias_ih_l0_reverse :  torch.Size([400])
lstm.bias_hh_l0_reverse :  torch.Size([400])
multihead_attn.in_proj_weight :  torc